# Decoration Detection

Minimal notebook using `ManuscriptPage` and `DecorationModel` from `pipeline.decorations`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.decorations import ManuscriptPage, DecorationModel

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
image_name = "MS-GG-00001-00001-000-00021_double_more_complex_filigranes.jpg"
image_name = "MS-GG-00001-00001-000-00726_double_many_figures.jpg"
image_name = "MS-GG-00001-00001-000-00175_double_packed_center.jpg"
image_name = "MS-GG-00001-00001-000-01185_double_elements_stain.jpg"
# Path to the source image that was processed by the pipeline
IMAGE_PATH = Path("../data/exemplars/" + image_name)

BACKEND      = "medieval_yolo"   # 'medieval_yolo' | 'yolo' | 'florence2' | 'florence2_medieval'
SIZE         = "x"               # 'n' | 's' | 'm' | 'l' | 'x'  (medieval_yolo only)
CONFIDENCE   = 0.20

REMOVE_LINES    = False          # paint out text-line regions before detection
USE_FIGURE_MASK = False          # restrict detection to figure_binary regions

In [ ]:
page = ManuscriptPage(IMAGE_PATH)
print(page)
print(f"Binding: {page.binding_side}  |  skew: {page.deskew_angle:.2f}°")
print(f"Lines: {page.n_lines}  |  Figures: {page.n_figures}")

In [ ]:
page.plot_raw()

In [ ]:
model = DecorationModel(backend=BACKEND, size=SIZE, confidence_threshold=CONFIDENCE)

In [ ]:
detections = model.run(page, remove_lines=REMOVE_LINES, use_figure_mask=USE_FIGURE_MASK)
print(f"{len(detections)} detections")

In [ ]:
page.plot_detections(detections)

In [ ]:
page.plot_summary(detections)

In [ ]:
page.summarize(detections)

### Batch processing all files

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Path to the source image that was processed by the pipeline
FOLDER_PATH = Path("../data/all_images/")

BACKEND      = "medieval_yolo"   # 'medieval_yolo' | 'yolo' | 'florence2' | 'florence2_medieval'
SIZE         = "x"               # 'n' | 's' | 'm' | 'l' | 'x'  (medieval_yolo only)
CONFIDENCE   = 0.20

REMOVE_LINES    = False          # paint out text-line regions before detection
USE_FIGURE_MASK = False          # restrict detection to figure_binary regions

# Loading fine tuned models
BACKEND = Path("/Users/luissalamanca/Dropbox/My_stuff/05_SDSCresearch/10_SideProjects/00_MedievalCambridge/line_counting/notebooks/runs/detect/cambridge-medieval-v1/weights/best.pt")
RESULTS_FOLDER = Path("../results/all_images_decorations-v1/")
RESULTS_FOLDER.mkdir(exist_ok=True, parents=True)


In [ ]:
model = DecorationModel(backend=BACKEND, size=SIZE, confidence_threshold=CONFIDENCE)

In [ ]:
all_images = sorted(FOLDER_PATH.glob("*.jpg"))
for img_path in all_images:
    print(f"\nProcessing {img_path.name}...")
    page = ManuscriptPage(img_path)
    detections = model.run(page, remove_lines=REMOVE_LINES, use_figure_mask=USE_FIGURE_MASK, save_json=True, save_dir=RESULTS_FOLDER)
    print(f"  {len(detections)} detections")
    page.plot_detections(detections, save_dir = RESULTS_FOLDER, exclude_categories = {"other", "other_interesting", "marginalia", "initial_filigree"})
    print(f"  Saved annotated image to: {RESULTS_FOLDER / img_path.name}")

In [ ]:
# With tqdm and no printing
from tqdm import tqdm
all_images = sorted(FOLDER_PATH.glob("*.jpg"))
for img_path in tqdm(all_images):
    page = ManuscriptPage(img_path)
    detections = model.run(page, remove_lines=REMOVE_LINES, use_figure_mask=USE_FIGURE_MASK, save_json=True, save_dir=RESULTS_FOLDER)
    page.plot_detections(detections, save_dir = RESULTS_FOLDER, exclude_categories = {"other", "other_interesting", "marginalia", "initial_filigree"}, flag_print=False)

### Evaluation — Precision, Recall, F1 against validated counts

In [ ]:
import pandas as pd

CSV_PATH = Path("../results/all_images_decorations-v1/validation_decorations.csv")

df = pd.read_csv(CSV_PATH)

# Derive categories from column names
cats = [c.replace("extracted_", "") for c in df.columns if c.startswith("extracted_")]

# Keep only validated rows (all validated columns must be filled)
val_cols = [f"validated_{cat}" for cat in cats]
df_val = df[df[val_cols].notna().all(axis=1)].copy()

print(f"Total pages    : {len(df)}")
print(f"Validated pages: {len(df_val)}")
print()

# Count-based TP / FP / FN per category
# TP_i = min(extracted_i, validated_i)  — correctly detected items
# FP_i = max(0, extracted_i - validated_i)  — extra detections
# FN_i = max(0, validated_i - extracted_i)  — missed items
rows = []
total_tp = total_fp = total_fn = 0

for cat in cats:
    ext = df_val[f"extracted_{cat}"].astype(int)
    val = df_val[f"validated_{cat}"].astype(int)

    tp = pd.concat([ext, val], axis=1).min(axis=1).sum()
    fp = (ext - val).clip(lower=0).sum()
    fn = (val - ext).clip(lower=0).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    rows.append({
        "category":  cat,
        "TP": int(tp), "FP": int(fp), "FN": int(fn),
        "Precision": round(precision, 3),
        "Recall":    round(recall, 3),
        "F1":        round(f1, 3),
    })
    total_tp += tp
    total_fp += fp
    total_fn += fn

# Micro-averaged overall (pools counts across all categories)
p_micro  = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
r_micro  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
f1_micro = 2 * p_micro * r_micro / (p_micro + r_micro) if (p_micro + r_micro) > 0 else 0.0

rows.append({
    "category":  "OVERALL (micro)",
    "TP": int(total_tp), "FP": int(total_fp), "FN": int(total_fn),
    "Precision": round(p_micro, 3),
    "Recall":    round(r_micro, 3),
    "F1":        round(f1_micro, 3),
})

pd.DataFrame(rows).set_index("category")